In [1]:
import copy
import pprint
import numpy as np
from enum import pickle_by_enum_name

from httpcore import TimeoutException
from selenium.webdriver.support.wait import WebDriverWait

import data
import time
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions
from selenium.webdriver.common.by import By
import time
import re

In [2]:
class Vendor:
    __lastId = 0

    def __init__(self, name, url, shipping):
        self.name = name
        self.url = url
        self.shipping = shipping
        self.id = Vendor.__lastId
        Vendor.__lastId += 1

class Listing:
    def __init__(self, name, url, price, vendor= None):
        self.name = name
        self.url = url
        self.price = price
        self.vendor = vendor


In [3]:
def url_build(tcg_id):
    url = "https://www.tcgplayer.com/product/{}?".format(tcg_id)
    url += "&Language=English"
    url += "&Condition=Near+Mint|Lightly+Played"
    return url

In [4]:
# Get card processed card data from scryfall
card_data = data.init_cards()
with open("b_deck.txt", encoding="utf8") as f:
    lines = f.readlines()
card_names = []
# strip amounts of cards, We assumed only 1 of each card
for line in lines:
    card_names.append(line[1:].strip())
cards = {}
for card_name in card_names:
    try:
        cards[card_name] = card_data[card_name]
    except KeyError as e:
        pprint.pprint(e)
        print("failed to find card" + card_name)
        exit(1)

card_listings = {}
vendor_id_counter = 0
vendors = {}  # dict containing TCG vendors

#driver init

In [5]:
#driver init
driver = webdriver.Firefox()
driver.implicitly_wait(3)
url = url_build(196523)
driver.get(url)
time.sleep(2)
driver.find_elements(By.CLASS_NAME, "tcg-input-select__trigger-container")[1].click()
time.sleep(1)
drop_downs = driver.find_elements(By.CLASS_NAME, "tcg-base-dropdown__item-content")
for drop_down in drop_downs:
    if drop_down.text == "50":
        drop_down.click()


In [6]:
def read_card(card, vendors, driver):
    prices = []
    for printing in card:
        prices.append(printing["price"])
    price_min, price_max = min(prices), max(prices)
    if price_min > 10:
        price_cutoff = price_min * 1.3 + 1
    else:
        price_cutoff = max(price_min * 1.3, price_min + 0.3)
    index_to_delete = []
    for idx, price in enumerate(prices):
        if price > price_cutoff:
            index_to_delete.append(idx)
    index_to_delete.sort(reverse=True)
    for idx in index_to_delete:
        card.pop(idx)
    if len(card) > 15:
        card = card[:15]
    listings = []

    for printing in card:
        url = url_build(printing["tcgplayer_id"])
        driver.get(url)
        time.sleep(0.5)
        #try:
        #    WebDriverWait(driver, 3).until(
        #        expected_conditions.presence_of_element_located((By.CLASS_NAME, "listing-item"))
        #    )
        #except TimeoutException:
        #    print("Failed to find listings for printing:\n{}".format(printing))
        #    continue
        elements = driver.find_elements(By.CLASS_NAME, "listing-item")

        for element in elements:

            price_element = element.find_element(By.CLASS_NAME, "listing-item__listing-data__info")

            match = re.search(r"Over \$\d+", price_element.text)
            shipping_type = None
            if match:
                shipping_type = int(match.group().split(" ")[1][1:])
            shipping = 0
            match = re.search(r"\$\d+\.\d{2}\sShipping[^:]", price_element.text + " ")
            if match:
                shipping = float(match.group().split(" ")[0][1:])
            price = float(price_element.find_element(By.CLASS_NAME, "listing-item__listing-data__info__price").text[1:].replace(',', ''))
            if not shipping_type:
                price += shipping
                shipping_type = 0
            elif shipping_type == 50:
                continue

            vendor = element.find_element(By.CLASS_NAME, "seller-info__name")
            vendor_name = vendor.text
            vendor_url = vendor.get_attribute("href")
            if vendor_name not in vendors:
                vendors[vendor_name] = Vendor(vendor_name, vendor_url, shipping)
                print("adding new vendor:{}".format(vendor_name))
            #print(vendor_name, vendor_url, price, shipping, shipping_type)
            listing = Listing(printing["name"], url, price, vendors[vendor_name])
            listings.append(listing)


    return listings

In [7]:
for card in cards:
    card_listings[card] = read_card(cards[card], vendors, driver)

adding new vendor:Tier Two Hobby
adding new vendor:Goldnugget
adding new vendor:FIX Gaming
adding new vendor:Turtle Nation
adding new vendor:BubbaD Cards
adding new vendor:LuxuryCollectorGuild
adding new vendor:SquirtleMVP
adding new vendor:Mississippi Heat TCG
adding new vendor:J-E Collectibles
adding new vendor:dubble tap
adding new vendor:AmpedMTG
adding new vendor:Paper Chase
adding new vendor:Basic Mountain Games
adding new vendor:Strategic Brew
adding new vendor:CardsBear
adding new vendor:Everything Geek
adding new vendor:Missed Lethal Games
adding new vendor:Central Valley Store
adding new vendor:ARCollectorsGuildLLC
adding new vendor:Landsraad Exchange
adding new vendor:GoblinShoeBox
adding new vendor:NostalgiAtog
adding new vendor:DAEVcards
adding new vendor:ARoguesDen
adding new vendor:Enchanted Bazaar
adding new vendor:The Media Market LLC
adding new vendor:ScavengerGroundGames
adding new vendor:DMV Games
adding new vendor:Breakhousecollects
adding new vendor:Pocket Rare
ad

In [8]:
print(len(vendors))
print(len(card_listings))
s = 0
for c in card_listings:
    s += len(card_listings[c])
print(s)
#pprint.pprint(vendors)

809
6
1547


In [9]:
class State:
    def __init__(self, under_five, card_to_store, shipping_covered=None, price=0):
        if shipping_covered is None:
            shipping_covered = set()
        #shipping_covered = [False] * len(under_five)
        #card_to_store = []
        self.shipping_covered = shipping_covered
        self.under_five = under_five
        self.card_to_store = card_to_store
        self.price = price

    def add_card(self, listing, vendor, price):
        """
        cases
        1. vendor in shipping covered
            do nothing
        2. vendor not in shipping covered and not in under_five
            add card to under five
            check if over 5
                move to shipping covered
        3.
        """
        shipping_covered = self.shipping_covered.copy()
        under_five = self.under_five.copy()
        #card_to_store = copy.deepcopy(self.card_to_store)
        card_to_store = self.card_to_store.copy()
        #if shipping_covered[vendor.id]:
        #    pass
        if vendor.id in shipping_covered:
            pass
        else:
            price += 1.3
            under_five[vendor.id] += price
            if under_five[vendor.id] >= 5:
                #shipping_covered[vendor.id] = True
                shipping_covered.add(vendor.id)
                price -= 1.3
        #print(len(card_to_store))
        #card_to_store[vendor.id].append(listing)
        card_to_store.append(listing)

        return State(under_five, card_to_store, shipping_covered, self.price + price)

In [10]:
import math
def optimize(vendors, card_listings):
    v = set()
    v2 = []
    for vendor in vendors:
        #print(vendors[vendor].id)
        #vendors[vendor].id -= 961
        v.add(vendors[vendor].id)
        v2.append(vendors[vendor].id)
    v2.sort()
    print(v2[0:100])
    print(len(vendors))
    print(len(v))
    #return
    temp =[]
    for x in range (len(vendors)):
        temp.append([])
    #print([[] for x in range(len(vendors))])
    options = [State([0] * len(vendors), [])]
    vendor_score = [0] * len(vendors)
    #print(card_listings)
    for card in card_listings:
        for listing in card_listings[card]:
            #print(listing)
            #print(listing.vendor.id)
            vendor_score[listing.vendor.id] += 1
    print(vendor_score)
    vendor_score = [x * 0.02 for x in vendor_score]
    for card in card_listings:
        card_listings[card].sort(key=lambda x: x.price - vendor_score[x.vendor.id])
    w = 0
    #card_listings.pop("Dark Ritual")
    #card_listings.pop("Timeline Culler")
    for x in card_listings.keys():
        y = card_listings[x]
        print(x)
        print(len(y))
        c = y[0]
        p = c.price
    keys = sorted(card_listings.keys(), key=lambda x: card_listings[x][0].price, reverse=True)
    for card in keys:

        if w == 5:
            pass
            #break
        w +=1
        new_options = []
        start = time.time()
        for option in options:
            cheapest_shipping_covered = False
            under_five = 2
            if w > 50:
                fresh  = 1
            else:
                fresh = 2
            for listing in card_listings[card]:
                price = listing.price
                vendor = listing.vendor
                #if price > 5 or option.shipping_covered[vendor.id]:
                if price > 5 or vendor.id in option.shipping_covered:
                    if not cheapest_shipping_covered:
                        new_options.append(option.add_card(listing, vendor, price))
                        cheapest_shipping_covered = True

                elif option.under_five[vendor.id] and under_five:
                    under_five -= 1
                    new_options.append(option.add_card(listing, vendor, price))
                elif fresh:
                    new_options.append(option.add_card(listing, vendor, price))
                    fresh -= 1
                elif not cheapest_shipping_covered and not under_five and not fresh:
                    break
        options = new_options
        end = time.time()
        print("time taken: {} Option size: {}".format(end - start, len(options)))
        if end - start > 0.5:
            options.sort(key=lambda o: o.price)
            options = options[:len(options) // 10]
            # cull
    options.sort(key=lambda o: o.price)
    print(options[0].price)
    print(options[0].card_to_store)
    for listing in options[0].card_to_store:
        print("card: {}, price: {}, vendor: {} ,link {}".format(listing.name, listing.price, listing.vendor.name, listing.url,))
            #pprint.pprint(array)
    return options[0]
option = optimize(vendors, card_listings)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99]
809
809
[1, 2, 2, 5, 1, 7, 2, 2, 1, 2, 6, 1, 1, 2, 4, 14, 3, 1, 1, 3, 1, 1, 2, 2, 3, 2, 15, 1, 2, 8, 2, 1, 2, 1, 10, 2, 1, 1, 2, 1, 1, 8, 6, 8, 4, 5, 2, 6, 9, 5, 4, 11, 2, 5, 2, 5, 3, 1, 3, 3, 2, 6, 6, 6, 4, 1, 1, 3, 6, 2, 1, 5, 8, 4, 1, 5, 4, 6, 1, 5, 1, 1, 2, 1, 2, 1, 1, 2, 1, 7, 1, 1, 1, 2, 1, 2, 2, 1, 1, 1, 2, 1, 1, 1, 3, 2, 1, 3, 1, 1, 1, 12, 1, 6, 2, 1, 2, 1, 1, 1, 5, 2, 2, 6, 4, 2, 1, 1, 1, 8, 1, 1, 2, 2, 1, 1, 1, 2, 1, 6, 1, 1, 8, 1, 1, 4, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 1, 3, 10, 5, 5, 6, 2, 2, 7, 2, 1, 6, 1, 5, 4, 2, 9, 6, 1, 3, 7, 1, 3, 11, 9, 1,

In [11]:
import pickle
with open("raw", "wb") as f:
    pickle.dump([option, vendors], f)

In [12]:
import pickle
with open("raw", "rb") as f:
    option, vendors = pickle.load(f)

In [13]:
from selenium.webdriver import ActionChains


def buy(option, driver):
    for listing in option.card_to_store:
        driver.get(listing.url)
        time.sleep(0.5)
        elements = driver.find_elements(By.CLASS_NAME, "listing-item")

        for element in elements:
            price_element = element.find_element(By.CLASS_NAME, "listing-item__listing-data__info")

            match = re.search(r"Over \$\d+", price_element.text)
            shipping_type = None
            if match:
                shipping_type = int(match.group().split(" ")[1][1:])
            if shipping_type == 50:
                continue

            vendor = element.find_element(By.CLASS_NAME, "seller-info__name")
            vendor_name = vendor.text
            if vendor_name == listing.vendor.name:
                add_element = element.find_element(By.CLASS_NAME, "add-to-cart")
                print("found listing")
                button = add_element.find_element(By.TAG_NAME, "button")
                header = driver.find_element(By.CLASS_NAME, "horizontal-filters-bar")
                driver.execute_script("""var element = arguments[0];element.parentNode.removeChild(element); """, header)
                driver.execute_script("arguments[0].scrollIntoView();", button)
                #time.sleep(1)
                ActionChains(driver).scroll_to_element(button).perform()
                print(button.text)
                print(button.get_attribute('innerHTML'))
                print(button)


                #return

                button.click()
                driver.find_element(By.CLASS_NAME, "tcg-snackbar__message")
                #return


                break
buy(option, driver)

found listing
Add to Cart
<!----><span class="tcg-standard-button__content">Add to Cart</span><!----><!---->
<selenium.webdriver.remote.webelement.WebElement (session="763194cc-7f0f-4836-a84b-701cbccbb3c4", element="25e033d9-2062-4ae7-a13b-c287c9fe69f1")>
found listing
Add to Cart
<!----><span class="tcg-standard-button__content">Add to Cart</span><!----><!---->
<selenium.webdriver.remote.webelement.WebElement (session="763194cc-7f0f-4836-a84b-701cbccbb3c4", element="cd6c3a99-ee1d-4f86-bb2e-52f59954aeaa")>
found listing
Add to Cart
<!----><span class="tcg-standard-button__content">Add to Cart</span><!----><!---->
<selenium.webdriver.remote.webelement.WebElement (session="763194cc-7f0f-4836-a84b-701cbccbb3c4", element="c35057e0-9930-4cdd-aab0-5da6d427492d")>
found listing
Add to Cart
<!----><span class="tcg-standard-button__content">Add to Cart</span><!----><!---->
<selenium.webdriver.remote.webelement.WebElement (session="763194cc-7f0f-4836-a84b-701cbccbb3c4", element="47a4bed2-31e9-4d

In [43]:
#manually go to cart. run this to see what cards are missing if any
print(len(option.card_to_store))
missing = []
elements = driver.find_elements(By.CLASS_NAME, "name")
b = []
x = "This is a sentence. (once a day) [twice a day]"
for e in elements:
    b.append(re.sub("[\(\[].*?[\)\]]", "", e.text).strip())

for listing in option.card_to_store:
    if listing.name not in b:
        print(listing.name)

<>:7: SyntaxWarning: invalid escape sequence '\('
<>:7: SyntaxWarning: invalid escape sequence '\('
C:\Users\alexk\AppData\Local\Temp\ipykernel_12336\1706658257.py:7: SyntaxWarning: invalid escape sequence '\('
  b.append(re.sub("[\(\[].*?[\)\]]", "", e.text).strip())


88
Seething Song
Poppet Stitcher // Poppet Factory
Bloodghast
Séance Board
Exsanguinate
Baral and Kari Zev
Ringsight
